# 03 — Auditoría de equidad algorítmica

**Tesis:** Estratificación de riesgo clínico consciente de la equidad — sesgo geográfico entre entidades federativas.

Este notebook es **el corazón de la contribución**. Toma el modelo de referencia (notebook 02) y pregunta no si acierta, sino **si acierta por igual** entre entidades, sexos y grupos de edad. Se calculan métricas de equidad estándar (equalized odds, paridad demográfica, calibración por grupo) de forma transparente (sin caja negra), y se explora el nivel interseccional (edad × entidad).

> Nota: las métricas se implementan a mano para que el cálculo sea auditable; son equivalentes a las de `Fairlearn`/`AIF360`, que pueden usarse como validación cruzada.


## 1. Datos y modelo

In [1]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

import tesis_common as tc

# El modelo NO se reentrena aquí: se carga el paquete que produjo el notebook 02.
# Antes este notebook entrenaba su propio HistGradientBoosting, así que "el modelo
# de referencia" y "el modelo auditado" eran objetos distintos que solo coincidían
# por compartir semilla — y en el notebook 04 ni eso (usaba otros hiperparámetros).
df, manifiesto = tc.cargar_cohorte()
paquete = tc.cargar_modelo(n_filas_cohorte=len(df))

X = df[tc.PREDICTORES].astype("float32")
y = df[tc.TARGET].astype(int)

# grupo_edad se toma del parquet (cortes definidos en tesis_common) en lugar de
# recalcularse: este notebook usaba bordes [-1, 17, 39, 59, 200] frente a los
# [-0.1, 17, 39, 59, 120] del notebook 01, dos definiciones del mismo atributo.
sens = df[["entidad_nombre", "sexo_mujer", "grupo_edad"]].copy()

idx_tr, idx_cal, idx_te = tc.particionar(y)
X_te = X.iloc[idx_te]
y_te = y.iloc[idx_te]
s_te = sens.iloc[idx_te].reset_index(drop=True)

# Riesgo CALIBRADO: es el que se despliega y el único cuya calibración por grupo
# tiene sentido auditar. Con el riesgo crudo, el error de calibración por entidad
# quedaba dominado por el sesgo global de class_weight="balanced" (~4x) y no por
# la inequidad geográfica, que es el objeto de estudio.
# Se conserva el puntaje SIN calibrar para cuantificar, en la seccion 8, cuanto
# del "error de calibracion por grupo" era en realidad el sesgo global de
# class_weight="balanced" y no inequidad entre entidades.
p_crudo = paquete["modelo"].predict_proba(X_te)[:, 1]
p = tc.puntuar(paquete, X_te)
y_te_a = y_te.to_numpy()

print("Modelo cargado:", tc.MODELO_PATH, "| hparams:", paquete["hparams"])
print("Test:", X_te.shape, "| AUROC global:", round(roc_auc_score(y_te_a, p), 4))
print("Riesgo medio predicho:", round(float(p.mean()), 4),
      "| mortalidad observada:", round(float(y_te_a.mean()), 4))

Modelo cargado: modelo_hgb_mortalidad.joblib | hparams: {'max_iter': 300, 'learning_rate': 0.05, 'max_depth': 6, 'class_weight': 'balanced', 'random_state': 42}
Test: (631663, 13) | AUROC global: 0.9503
Riesgo medio predicho: 0.0558 | mortalidad observada: 0.056


## 2. Umbral de operación

Las métricas de equalized odds dependen del umbral. Se fija el umbral para lograr una **sensibilidad global objetivo del 80%** (adecuada para una herramienta de triage), y se mide cómo se reparte esa sensibilidad entre grupos.


In [2]:
# Mismo punto de operación que los notebooks 02 y 04 (tesis_common).
THR, tpr_obj, fpr_obj = tc.umbral_para_sensibilidad(y_te_a, p)
print(f"Umbral = {THR:.4f}  ->  TPR global {tpr_obj:.3f}, FPR global {fpr_obj:.3f}")

Umbral = 0.1562  ->  TPR global 0.795, FPR global 0.059


## 3. Funciones de equidad (transparentes)

In [3]:
# Las métricas de equidad tienen una sola implementación, en tesis_common, que
# comparten los notebooks 02, 03 y 04. Antes cada notebook definía su propia
# variante (`metricas_por_grupo` aquí, `tf`/`gaps` en el 04) con criterios de
# soporte mínimo distintos, de modo que sus tablas no eran comparables.
metricas_por_grupo = tc.metricas_por_grupo
resumen_disparidad = tc.resumen_disparidad
grupos_evaluables = tc.grupos_evaluables

print("Criterio de soporte por grupo: n >=", tc.MIN_N_GRUPO,
      "y positivos >=", tc.MIN_POS_GRUPO)
print(metricas_por_grupo.__doc__.strip().splitlines()[0])

Criterio de soporte por grupo: n >= 500 y positivos >= 20
Métricas de desempeño y equidad por grupo, a un umbral común.


## 4. Equidad por ENTIDAD FEDERATIVA (eje central)

In [4]:
tab_ent = metricas_por_grupo(y_te_a, p, s_te["entidad_nombre"], THR).sort_values("AUROC")
_, ent_descartadas = grupos_evaluables(s_te["entidad_nombre"], y_te_a)

print(pd.concat([tab_ent.head(5), tab_ent.tail(5)])[
    ["grupo", "n", "tasa", "AUROC", "TPR", "FPR", "cal_pred", "cal_obs"]].round(3).to_string(index=False))
print("\n", resumen_disparidad(tab_ent).to_string())

print(f"\nEntidades evaluadas: {len(tab_ent)} de 32")
if len(ent_descartadas):
    print("Excluidas por soporte insuficiente:")
    print(ent_descartadas.to_string(index=False))

              grupo     n  tasa  AUROC   TPR   FPR  cal_pred  cal_obs
          CHIHUAHUA 11072 0.071  0.921 0.688 0.081     0.066    0.071
            CHIAPAS  3831 0.061  0.927 0.662 0.065     0.057    0.061
BAJA CALIFORNIA SUR 11712 0.031  0.927 0.683 0.040     0.035    0.031
     AGUASCALIENTES  5498 0.061  0.930 0.657 0.060     0.053    0.061
            DURANGO  6297 0.057  0.931 0.631 0.052     0.047    0.057
       QUINTANA ROO 12151 0.041  0.962 0.813 0.041     0.039    0.041
             SONORA 16698 0.056  0.963 0.868 0.063     0.062    0.056
    SAN LUIS POTOSI 17326 0.046  0.965 0.868 0.057     0.053    0.046
           GUERRERO 12589 0.067  0.968 0.925 0.080     0.075    0.067
            TABASCO 24620 0.022  0.977 0.879 0.036     0.035    0.022

 gap_TPR (equalized odds)           0.2935
gap_FPR                            0.0867
gap_AUROC                          0.0563
gap_seleccion (paridad demogr.)    0.1556
max_error_calibracion              0.0231

Entidades evaluad

**Lectura:** una brecha grande de TPR significa que la sensibilidad del modelo —su capacidad de detectar a quien va a fallecer— depende de la entidad de residencia; es decir, el modelo "protege" mejor a unos estados que a otros.

La columna de calibración se mide ahora sobre el **riesgo calibrado**. Esto importa metodológicamente: con el riesgo crudo, el error de calibración por entidad estaba dominado por la sobreestimación global de `class_weight="balanced"` (riesgo medio predicho ≈ 0.205 frente a 0.056 observado), de modo que la tabla medía un artefacto del entrenamiento y no la inequidad geográfica. Tras la calibración global, lo que queda es el **error residual de calibración por entidad**, que sí es atribuible al grupo.

Las brechas de TPR, FPR y AUROC no se ven afectadas por la calibración, porque la transformación isotónica es monótona y el umbral se fija por sensibilidad objetivo: preserva el ordenamiento y, con él, todas las métricas basadas en ranking.


## 5. Equidad por SEXO y por GRUPO DE EDAD

In [5]:
tab_sexo = metricas_por_grupo(y_te_a, p, s_te["sexo_mujer"].map({1:"Mujer",0:"Hombre"}), THR)
print("== SEXO =="); print(tab_sexo[["grupo","n","tasa","AUROC","TPR","FPR","seleccion"]].round(3).to_string(index=False))
print(resumen_disparidad(tab_sexo).to_string())

tab_edad = metricas_por_grupo(y_te_a, p, s_te["grupo_edad"].astype(str), THR)
print("\n== GRUPO DE EDAD =="); print(tab_edad[["grupo","n","tasa","AUROC","TPR","FPR","seleccion"]].round(3).to_string(index=False))
print(resumen_disparidad(tab_edad).to_string())

== SEXO ==
 grupo      n  tasa  AUROC   TPR   FPR  seleccion
Hombre 309437 0.068  0.946 0.811 0.073      0.123
 Mujer 322226 0.044  0.954 0.772 0.046      0.079
gap_TPR (equalized odds)           0.0393
gap_FPR                            0.0268
gap_AUROC                          0.0082
gap_seleccion (paridad demogr.)    0.0448
max_error_calibracion              0.0005



== GRUPO DE EDAD ==
grupo      n  tasa  AUROC   TPR   FPR  seleccion
 0-17  47676 0.003  0.935 0.266 0.003      0.004
18-39 303100 0.009  0.935 0.608 0.015      0.020
40-59 196593 0.056  0.904 0.743 0.062      0.100
  60+  84285 0.256  0.844 0.849 0.306      0.445
gap_TPR (equalized odds)           0.5824
gap_FPR                            0.3026
gap_AUROC                          0.0903
gap_seleccion (paridad demogr.)    0.4408
max_error_calibracion              0.0008


**Contraste esperado:** el sesgo por **sexo** suele ser pequeño; el de **edad** es grande pero en buena parte mecánico (la edad es el predictor dominante y las tasas base difieren mucho). El hallazgo original de la tesis es el sesgo **geográfico**, que no se explica solo por composición demográfica (ver sección 7).

## 6. ¿Con qué se asocia la inequidad geográfica?

Se correlaciona la brecha por entidad con proxies internos de desigualdad estructural (tamaño de muestra y tasa de mortalidad). En una fase posterior se integrarán índices externos (marginación CONAPO, camas por habitante).


In [6]:
from scipy import stats

# Se reportan r, n y p: una correlación sobre 32 puntos sin significancia no
# sustenta la afirmación "el sesgo no se explica por X", que es una conclusión
# de la discusión. Se añade Spearman por robustez a no linealidad y atípicos.
CORRELATOS = {}

def correlacion(a, b, clave, etiqueta):
    """Calcula Pearson y Spearman con n y p, los imprime y los acumula."""
    m = a.notna() & b.notna()
    r, pr = stats.pearsonr(a[m], b[m])
    rho, ps = stats.spearmanr(a[m], b[m])
    CORRELATOS[clave] = {"r": round(float(r), 3), "p_r": round(float(pr), 3),
                         "rho": round(float(rho), 3), "p_rho": round(float(ps), 3),
                         "n": int(m.sum()), "etiqueta": etiqueta}
    print(f"{etiqueta:38s} r={r:+.3f} (p={pr:.3f})   rho={rho:+.3f} (p={ps:.3f})   n={int(m.sum())}")

correlacion(tab_ent["AUROC"], tab_ent["n"],    "auroc_vs_n",     "AUROC vs tamano de muestra")
correlacion(tab_ent["AUROC"], tab_ent["tasa"], "auroc_vs_tasa",  "AUROC vs tasa de mortalidad")
correlacion(tab_ent["TPR"],   tab_ent["n"],    "tpr_vs_n",       "TPR   vs tamano de muestra")
correlacion(tab_ent["TPR"],   tab_ent["tasa"], "tpr_vs_tasa",    "TPR   vs tasa de mortalidad")

# La integración del índice de marginación CONAPO se hace en el notebook 04,
# donde vive el catálogo (tesis_common.CONAPO_NORM) junto con su advertencia de
# procedencia sin verificar.

AUROC vs tamano de muestra             r=+0.194 (p=0.288)   rho=+0.293 (p=0.104)   n=32
AUROC vs tasa de mortalidad            r=-0.373 (p=0.036)   rho=-0.335 (p=0.061)   n=32
TPR   vs tamano de muestra             r=+0.024 (p=0.896)   rho=+0.123 (p=0.503)   n=32
TPR   vs tasa de mortalidad            r=+0.165 (p=0.366)   rho=+0.158 (p=0.388)   n=32


## 7. Interseccionalidad — equidad dentro de los adultos mayores (60+)

In [7]:
# El criterio general de soporte (n>=500) dejaria fuera entidades justo en el
# subgrupo donde interesa mirar, asi que aqui se relaja a 300. Se reporta cual es
# la diferencia para que el lector juzgue: la conclusion no debe depender de eso.
MIN_N_INTERSECCIONAL = 300

m60 = (s_te["grupo_edad"] == "60+").to_numpy()
tab_60 = metricas_por_grupo(y_te_a[m60], p[m60],
                            s_te.loc[s_te["grupo_edad"]=="60+", "entidad_nombre"],
                            THR, min_n=MIN_N_INTERSECCIONAL, min_pos=20).sort_values("AUROC")
print(pd.concat([tab_60.head(3), tab_60.tail(3)])[["grupo","n","tasa","AUROC","TPR"]].round(3).to_string(index=False))
print("\nAun controlando por edad avanzada, AUROC varía de",
      round(tab_60["AUROC"].min(),3), "a", round(tab_60["AUROC"].max(),3),
      "| gap TPR:", round(tab_60["TPR"].max()-tab_60["TPR"].min(),3))

# Contraste con el criterio general, por transparencia.
tab_60_estandar = tab_60[tab_60["n"] >= tc.MIN_N_GRUPO]
bajo_criterio = tab_60[tab_60["n"] < tc.MIN_N_GRUPO]
rango_relajado = float(tab_60["AUROC"].max() - tab_60["AUROC"].min())
rango_estandar = float(tab_60_estandar["AUROC"].max() - tab_60_estandar["AUROC"].min())
print(f"\nCriterio relajado (n>={MIN_N_INTERSECCIONAL}): {len(tab_60)} entidades, "
      f"rango AUROC {rango_relajado:.3f}")
print(f"Criterio general  (n>={tc.MIN_N_GRUPO}): {len(tab_60_estandar)} entidades, "
      f"rango AUROC {rango_estandar:.3f}")
if len(bajo_criterio):
    print("Entidades por debajo del criterio general:")
    print(bajo_criterio[["grupo", "n", "tasa", "AUROC"]].round(3).to_string(index=False))

           grupo     n  tasa  AUROC   TPR
         CHIAPAS   482 0.322  0.688 0.748
       CHIHUAHUA  1798 0.274  0.737 0.732
         JALISCO  4212 0.382  0.758 0.771
        GUERRERO  2046 0.277  0.889 0.951
CIUDAD DE MEXICO 20942 0.128  0.895 0.858
         TABASCO  2801 0.117  0.942 0.930

Aun controlando por edad avanzada, AUROC varía de 0.688 a 0.942 | gap TPR: 0.256

Criterio relajado (n>=300): 32 entidades, rango AUROC 0.254
Criterio general  (n>=500): 31 entidades, rango AUROC 0.204
Entidades por debajo del criterio general:
  grupo   n  tasa  AUROC
CHIAPAS 482 0.322  0.688


## 8. Figuras

In [8]:
import json
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(17, 8))
t = tab_ent.sort_values("TPR")
ax[0].barh(t["grupo"], t["TPR"], color="#1F3864")
ax[0].axvline(tc.SENSIBILIDAD_OBJETIVO, color="red", ls="--", lw=1)
ax[0].set_title("Sensibilidad (TPR) por entidad — linea roja = objetivo global")
t2 = tab_ent.sort_values("AUROC")
ax[1].barh(t2["grupo"], t2["AUROC"], color="#2E5496"); ax[1].set_xlim(0.88, 1.0)
ax[1].set_title("AUROC por entidad")
ax[2].scatter(tab_ent["cal_obs"], tab_ent["cal_pred"], color="#1F3864")
lim = [0, max(tab_ent["cal_pred"].max(), tab_ent["cal_obs"].max()) * 1.1]
ax[2].plot(lim, lim, "r--")
ax[2].set_xlabel("mortalidad observada"); ax[2].set_ylabel("riesgo medio predicho")
ax[2].set_title("Calibracion por entidad (ideal = diagonal)")
plt.tight_layout(); plt.savefig("equidad_por_entidad.png", dpi=110)
print("Figura guardada: equidad_por_entidad.png")

# Contraste crudo vs calibrado del error de calibracion por entidad. El umbral no
# influye: cal_pred y cal_obs son promedios, no dependen del corte.
tab_ent_crudo = metricas_por_grupo(y_te_a, p_crudo, s_te["entidad_nombre"], THR)
err_cal_crudo = float((tab_ent_crudo["cal_pred"] - tab_ent_crudo["cal_obs"]).abs().max())
err_cal_calib = float((tab_ent["cal_pred"] - tab_ent["cal_obs"]).abs().max())
print(f"Error max. de calibracion por entidad: crudo {err_cal_crudo:.4f} -> calibrado {err_cal_calib:.4f}")

# Exportación para que el documento LaTeX cite exactamente estas cifras en vez de
# transcribirlas a mano, que es como se desincronizan los resultados del texto.
auditoria = {
    "umbral": round(float(THR), 4),
    "sensibilidad_objetivo": tc.SENSIBILIDAD_OBJETIVO,
    "auroc_global": round(float(roc_auc_score(y_te_a, p)), 4),
    "n_test": int(len(y_te_a)),
    "por_entidad": tab_ent.round(4).to_dict(orient="records"),
    "por_sexo": tab_sexo.round(4).to_dict(orient="records"),
    "por_grupo_edad": tab_edad.round(4).to_dict(orient="records"),
    "mayores_60_por_entidad": tab_60.round(4).to_dict(orient="records"),
    "interseccional_criterio": {
        "min_n_usado": MIN_N_INTERSECCIONAL,
        "min_n_general": tc.MIN_N_GRUPO,
        "n_entidades_relajado": int(len(tab_60)),
        "n_entidades_estandar": int(len(tab_60_estandar)),
        "rango_relajado": round(rango_relajado, 4),
        "rango_estandar": round(rango_estandar, 4),
        "peor_estandar": tab_60_estandar.sort_values("AUROC").iloc[0]["grupo"],
        "peor_estandar_auroc": round(float(tab_60_estandar["AUROC"].min()), 4),
        "bajo_criterio": bajo_criterio[["grupo", "n"]].to_dict(orient="records"),
    },
    "max_error_calibracion_crudo": round(err_cal_crudo, 4),
    "max_error_calibracion_calibrado": round(err_cal_calib, 4),
    "correlatos": CORRELATOS,
    "disparidad_entidad": resumen_disparidad(tab_ent).to_dict(),
    "disparidad_sexo": resumen_disparidad(tab_sexo).to_dict(),
    "disparidad_edad": resumen_disparidad(tab_edad).to_dict(),
}
with open("auditoria_equidad.json", "w", encoding="utf-8") as fh:
    json.dump(auditoria, fh, indent=2, ensure_ascii=False)
print("Resultados exportados: auditoria_equidad.json")

Figura guardada: equidad_por_entidad.png


Error max. de calibracion por entidad: crudo 0.2108 -> calibrado 0.0231
Resultados exportados: auditoria_equidad.json


## 9. Síntesis y próximos pasos (notebook 04 — mitigación)

**Conclusión de la auditoría:** el modelo, pese a su alto AUROC global, presenta inequidad medible entre entidades —brechas de sensibilidad y de calibración residual— que persiste incluso dentro del subgrupo de adultos mayores. Esto confirma empíricamente la hipótesis H1 y motiva la mitigación.

**Notebook 04:**
1. Integración del índice de marginación por entidad y su correlación con el sesgo.
2. Mitigación pre/in/post-procesamiento (reponderación por entidad, restricción de equalized odds, umbrales por grupo).
3. Frontera de Pareto equidad–desempeño.

> El recalibrado global ya no es tarea del 04: se aplica en el notebook 02, sobre un bloque de calibración propio, y todos los notebooks consumen el riesgo calibrado a través de `tesis_common.puntuar`.
